# Imported FMU in DPsim MNA: comparison with MATLAB

This notebook runs the **actual MATLAB-exported FMI 3.0 Model Exchange FMU** as
an electrical SSN branch inside DPsim's EMT/MNA topology, then compares the
results against two MATLAB references:

- **Trapezoidal:** the same integration rule as DPsim's general SSN component.
- **Simulink ode45:** a tightly toleranced reference with a different integrator.

The example obtains the branch equations from FMI evaluations. Its C++ topology
contains an imported branch and an external prescribed-voltage boundary; it uses
no dedicated resistor, inductor or capacitor classes and no FMI `doStep()`.

```text
           positive DPsim node
                 |       |
    voltage      |       |  imported FMU-backed
    boundary     |       |  SSN electrical branch
                 |       |
                 ground
```

Benchmark: 230 V RMS, 50 Hz, 0–0.1 s, 10 µs step, zero initial states in the
MATLAB export. These experiment settings are defined in `FMI_EMT_LinearSSN.cpp`.
Only continuous, homogeneous linear models are supported by this first importer.

**VS Code:** select the **DPsim Python** kernel and choose **Run All**. Adjust the
MATLAB results directory in the first cell if you move the exported files.

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys
import zipfile
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# The kernel normally starts in the VS Code workspace or notebook directory.
REPO_ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "dpsim-fmi" / "CMakeLists.txt").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        "Start the kernel inside the DPsim repository, or set REPO_ROOT manually."
    )

MATLAB_RESULTS = Path("/Volumes/SSDMatthias/rlc_fmu_test_20261015_170838")
FMU = MATLAB_RESULTS / "dpsim_rlc_me.fmu"
BUILD_DIR = REPO_ROOT / "build-macos-check"
RUN_DIR = REPO_ROOT / "outputs" / "fmi-notebook"
BUILD_EXAMPLE = False  # Set True to rebuild after changing C++ code.
VOLTAGE_INPUT = "u"
CURRENT_OUTPUT = "i"

RUN_DIR.mkdir(parents=True, exist_ok=True)
for name in (FMU.name, "reference_trapezoidal.csv", "reference_simulink.csv"):
    path = MATLAB_RESULTS / name
    if not path.is_file():
        raise FileNotFoundError(path)

print("Python:", sys.executable)
print("Repository:", REPO_ROOT)
print("FMU:", FMU)
print("Results:", RUN_DIR)

## 1. Inspect the exported interface

Variable names and state metadata come from the FMU. The electrical interpretation
is supplied explicitly: `u` is terminal voltage and `i` is current entering the
positive terminal. `v_C` is a diagnostic output.

In [ ]:
with zipfile.ZipFile(FMU) as archive:
    root = ET.fromstring(archive.read("modelDescription.xml"))
    binaries = [name for name in archive.namelist() if name.startswith("binaries/")]

assert root.attrib["fmiVersion"] == "3.0"
me = root.find("ModelExchange")
assert me is not None, "This benchmark requires a Model Exchange FMU."
print("Model:", root.attrib["modelName"])
print("Model Exchange capabilities:", me.attrib)
print("Packaged binaries:", binaries)
variables = pd.DataFrame(
    [
        {
            "name": v.get("name"),
            "type": v.tag,
            "causality": v.get("causality", "local"),
            "value_reference": v.get("valueReference"),
            "derivative_of": v.get("derivative", ""),
        }
        for v in root.find("ModelVariables")
    ]
)
display(variables)

## 2. Build if needed and run inside DPsim

The run is performed in `outputs/fmi-notebook`, so it does not overwrite results
from your terminal runs. Re-running the notebook replaces only this notebook's
own output files. On this Mac, the build command uses Command Line Tools locally
to match the existing DPsim build, leaving MATLAB's selected Xcode unchanged.

In [ ]:
exe_name = (
    "FMI_EMT_LinearSSN.exe" if platform.system() == "Windows" else "FMI_EMT_LinearSSN"
)
executable = BUILD_DIR / "dpsim-fmi" / exe_name
build_env = os.environ.copy()
clt = Path("/Library/Developer/CommandLineTools")
if platform.system() == "Darwin" and clt.is_dir():
    build_env["DEVELOPER_DIR"] = str(clt)


def run_checked(command, *, cwd, env=None):
    result = subprocess.run(
        [str(arg) for arg in command],
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode:
        print(result.stdout)
        result.check_returncode()
    return result.stdout


if BUILD_EXAMPLE or not executable.is_file():
    configuration = run_checked(
        [
            "cmake",
            "-S",
            REPO_ROOT,
            "-B",
            BUILD_DIR,
            "-DCMAKE_BUILD_TYPE=Release",
            "-DWITH_FMI=ON",
            "-DDPSIM_BUILD_EXAMPLES=ON",
            "-DFETCH_SPDLOG=ON",
            "-DFETCH_GRID_DATA=OFF",
            "-DDPSIM_BUILD_DOC=OFF",
            "-DWITH_VILLAS=OFF",
            "-DWITH_OPENMP=OFF",
        ],
        cwd=REPO_ROOT,
        env=build_env,
    )
    (RUN_DIR / "configure.log").write_text(configuration)
    build_log = run_checked(
        [
            "cmake",
            "--build",
            BUILD_DIR,
            "--parallel",
            "3",
            "--target",
            "FMI_EMT_LinearSSN",
        ],
        cwd=REPO_ROOT,
        env=build_env,
    )
    (RUN_DIR / "build.log").write_text(build_log)
    print(build_log)

run_log = run_checked(
    [executable, FMU, VOLTAGE_INPUT, CURRENT_OUTPUT],
    cwd=RUN_DIR,
)
(RUN_DIR / "run.log").write_text(run_log)
print(run_log)
DPSIM_CSV = RUN_DIR / "logs" / "fmi_rlc.csv"
assert DPSIM_CSV.is_file(), "DPsim did not produce a results file."

## 3. Load and align the reference results

MATLAB's `ode45` output can have different timestamps. Reference signals are
linearly interpolated onto the DPsim time grid, without extrapolation. The
trapezoidal reference already uses the same 10 µs grid.

In [ ]:
SIGNALS = {
    "u_V": "Terminal voltage [V]",
    "i_A": "Branch current [A]",
    "v_C_V": "Capacitor voltage [V]",
}


def read_results(path, *, dpsim=False):
    frame = pd.read_csv(path, skipinitialspace=True)
    frame.columns = frame.columns.str.strip()
    if dpsim:
        frame = frame.rename(columns={"time": "time_s", "i": "i_A", "v_C": "v_C_V"})
    required = ["time_s", *SIGNALS]
    if not set(required).issubset(frame.columns):
        raise ValueError(f"Missing columns in {path}: {frame.columns.tolist()}")
    frame = frame[required].astype(float)
    assert (
        len(frame) > 1 and np.isfinite(frame.to_numpy()).all()
    ), f"Invalid data: {path}"
    assert np.all(np.diff(frame.time_s) > 0), f"Times must increase: {path}"
    return frame


dpsim = read_results(DPSIM_CSV, dpsim=True)
references = {
    "MATLAB trapezoidal": read_results(MATLAB_RESULTS / "reference_trapezoidal.csv"),
    "MATLAB Simulink ode45": read_results(MATLAB_RESULTS / "reference_simulink.csv"),
}
time = dpsim.time_s.to_numpy()
assert len(time) == 10001
assert np.allclose(np.diff(time), 1e-5, rtol=0, atol=1e-12)
assert abs(time[0]) < 1e-12 and abs(time[-1] - 0.1) < 1e-12

aligned = {}
for name, reference in references.items():
    assert reference.time_s.iloc[0] <= time[0] + 1e-12
    assert reference.time_s.iloc[-1] >= time[-1] - 1e-12
    aligned[name] = pd.DataFrame(
        {
            "time_s": time,
            **{
                signal: np.interp(time, reference.time_s, reference[signal])
                for signal in SIGNALS
            },
        }
    )

display(
    pd.DataFrame(
        [
            {
                "dataset": "DPsim imported FMU",
                "samples": len(dpsim),
                "start_s": time[0],
                "end_s": time[-1],
            },
            *[
                {
                    "dataset": name,
                    "samples": len(ref),
                    "start_s": ref.time_s.iloc[0],
                    "end_s": ref.time_s.iloc[-1],
                }
                for name, ref in references.items()
            ],
        ]
    )
)

## 4. Compare the waveforms

The three curves should overlap closely. The next section magnifies their
numerical differences.

In [ ]:
plt.rcParams.update(
    {"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False}
)
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True, constrained_layout=True)
for ax, (signal, label) in zip(axes, SIGNALS.items()):
    ax.plot(
        time * 1e3,
        aligned["MATLAB Simulink ode45"][signal],
        color="#0072B2",
        label="MATLAB Simulink ode45",
        linewidth=1.8,
    )
    ax.plot(
        time * 1e3,
        aligned["MATLAB trapezoidal"][signal],
        color="#009E73",
        marker=".",
        markersize=3,
        markevery=180,
        linewidth=0.7,
        label="MATLAB trapezoidal",
    )
    ax.plot(
        time * 1e3,
        dpsim[signal],
        color="#D55E00",
        linestyle="--",
        linewidth=1.2,
        label="DPsim imported FMU",
    )
    ax.set_ylabel(label)
    ax.grid(alpha=0.2)
axes[0].legend(loc="upper right", ncol=3, fontsize=9)
axes[-1].set_xlabel("Time [ms]")
fig.suptitle("Imported RLC branch: DPsim MNA versus MATLAB")
fig.savefig(RUN_DIR / "waveform_comparison.png", dpi=180)
plt.show()

## 5. Quantify the differences

Errors are **DPsim minus MATLAB**, after time alignment. Each metric retains the
signal's physical unit. DPsim's current CSV logger rounds signal values to six
decimal places, so agreement within approximately 0.5 µA or 0.5 µV can be limited
by output formatting.

The trapezoidal comparison tests agreement between the equivalent discretizations.
Differences from `ode45` also include integration and reference interpolation
errors; they do not measure FMI import error alone.

In [ ]:
errors = {}
rows = []
units = {"u_V": "V", "i_A": "A", "v_C_V": "V"}
for name, reference in aligned.items():
    errors[name] = dpsim[list(SIGNALS)] - reference[list(SIGNALS)]
    for signal in SIGNALS:
        values = errors[name][signal].to_numpy()
        rows.append(
            {
                "reference": name,
                "signal": signal,
                "unit": units[signal],
                "max_abs_error": np.max(np.abs(values)),
                "RMSE": np.sqrt(np.mean(values**2)),
            }
        )
metrics = pd.DataFrame(rows)
display(metrics.style.format({"max_abs_error": "{:.9g}", "RMSE": "{:.9g}"}))
metrics.to_csv(RUN_DIR / "comparison_metrics.csv", index=False)

for name, frame in errors.items():
    exported = frame.copy()
    exported.insert(0, "time_s", time)
    label = "trapezoidal" if name == "MATLAB trapezoidal" else "simulink_ode45"
    exported.to_csv(RUN_DIR / f"errors_{label}.csv", index=False)

trap = metrics.loc[metrics.reference == "MATLAB trapezoidal", "max_abs_error"]
assert (trap < 6e-7).all(), "Trapezoidal agreement exceeds the CSV rounding allowance."
display(
    Markdown(
        "**PASS:** all 10,001 samples agree with the MATLAB trapezoidal reference within CSV rounding."
    )
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, constrained_layout=True)
for ax, signal, scale, unit in (
    (axes[0], "i_A", 1e6, "Current error [µA]"),
    (axes[1], "v_C_V", 1e3, "Capacitor-voltage error [mV]"),
):
    ax.plot(
        time * 1e3,
        errors["MATLAB Simulink ode45"][signal] * scale,
        color="#0072B2",
        label="DPsim − MATLAB ode45",
    )
    ax.plot(
        time * 1e3,
        errors["MATLAB trapezoidal"][signal] * scale,
        color="#D55E00",
        linewidth=0.8,
        label="DPsim − MATLAB trapezoidal",
    )
    ax.axhline(0, color="0.5", linewidth=0.6)
    ax.set_ylabel(unit)
    ax.grid(alpha=0.2)
axes[0].legend()
axes[-1].set_xlabel("Time [ms]")
fig.suptitle("Differences between DPsim and the MATLAB references")
fig.savefig(RUN_DIR / "error_comparison.png", dpi=180)
plt.show()
print("Saved plots, CSV metrics, error traces and the execution log in:", RUN_DIR)

## Interpretation and files

- The terminal voltage checks that both simulations use the same excitation and time origin.
- The current and capacitor voltage check the imported dynamics and diagnostic-output mapping.
- Matching the trapezoidal reference is the main importer/SSN check for this benchmark.
- The `ode45` difference shows the effect of the selected timestep and integration method.
- The notebook invokes the built C++ MNA example. It does not substitute a Python RLC simulation.
- The synthetic FMUs and hard-coded expectations in `ImportTest.cpp` are separate automated tests;
  they are not used to produce these waveforms.

Implementation: `dpsim-fmi/src/LinearModel.cpp` and `LinearSSN.cpp`.
Experiment: `dpsim-fmi/examples/FMI_EMT_LinearSSN.cpp`.
Build instructions and supported-model restrictions: `dpsim-fmi/README.md`.